## Data Laurel Carney
This notebook has a look at some of the properties of the data from Laurel Carney

In [ ]:
import sys
import os

# The root of the project is one folder down (and contains the python_utils folder)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add project root to sys.path if not already present
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import scipy.signal as sig
import numpy as np
import matplotlib.pyplot as plt
from utils_python import mp_utils as mp # matching pursuit stuff
from utils_python import filter_utils as fu

import librosa # common audio processing library
import IPython.display as ipd # For playing audio

PATH_CARNEY = "/home/dimme/Documents/Databases/carney_cat_revcor/CarneyEarlabRevcorData"
from matplotlib.ticker import LogLocator

%load_ext autoreload
%autoreload 2

## Load revcor filters
First all revcors are loaded. Note that the sample_rates are 20 kHz!!!! One of the files is discarded as it contains only zeros. 

In [ ]:
revcors, sample_rates =  fu.ReadDataCarney(PATH_CARNEY)
res = list(set(sample_rates))
print(f"The following sample rates are present: {res} [Hz]")


I was curious for the energy and the amplitudes of each revcor filter on  log-scale... Boxplots...

In [ ]:
norms = []
max_amp = []
for revcor in revcors:
    norms.append(np.log10(np.linalg.norm(revcor)))
    max_amp.append(np.log10(np.max(np.abs(revcor/norms[-1]))))

plt.figure()
plt.title("Boxplot of norms revcors")
plt.boxplot(norms)

plt.figure()
plt.title("Boxplot of amplitudes of length 1 normalised revcors")
plt.boxplot(max_amp)

plt.show()

## Compute filter statistics
By running `fu.process_data_Carney` some filter statistics are computed. Feel free to have a look at the function, it's in `python_utils/filter_utils.py` (or something like that). It returns:
 - the spectral entropy (measure of chaoticness),
 - the spectral centroid (measure of center frequency),
 - the spectral spread (measure of bandwidth),
 - the peak frequency (measure of center frequency).
 - the center frequency (another measure of center frequency, confusing right?)
 - the bandwidth_3db (the bandwidth where -3 dB is reached)

Note that how "trustworthy" all these metrics are is up for debate: the data is noisy...

The metrics are computed using: `fu.estimate_fir_params(filter_coeffs, fs = 1.0, plotting = False, Nfreqz = 2048, features=["centroid", "spread", "entropy"])` (this lists the defaults, but more features are available. filter_coeffs are the coefficients, i.e. a particular kernel would be filter_coeffs)

In [ ]:
#process_data_Carney(revcors, sample_rate = 20000, plotting=False, features=["centroid", "spread", "entropy", "peak_frequency", "bandwidth_3db"])
results, revcors_f = fu.process_data_Carney(revcors)

In [ ]:
# Measures for center frequency
peak_frequencies = [result["peak_frequency"] for result in results]
center_frequencies = [result["center_frequency"] for result in results]
centroid_frequencies = [result["centroid"] for result in results]

# measures for bandwidth
bandwidths_3db = [result["bandwidth_3db"] for result in results]
spreads = [result["spread"] for result in results]

# Spectral entropy
entropies = [result["entropy"] for result in results]

Create a number of scatterplots

In [ ]:
# Create the scatter plot for peak_frequency vs bandwidths_3db
plt.figure()
scatter = plt.scatter(
    x=peak_frequencies,
    y=bandwidths_3db,
    c=entropies,
    cmap="viridis",  
    alpha=0.7,       
)
plt.xscale('log')
plt.yscale('log')

ax = plt.gca()  # Get current axis

# Set custom tick positions and labels for x-axis
x_ticks = [100, 250, 500, 1000, 2000, 4000]
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_ticks)

# Set custom tick positions and labels for y-axis
y_ticks = [50, 100, 250, 500, 1000, 2000, 4000]
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_ticks)

# Add labels and colorbar
plt.xlabel("Peak Frequency [Hz]")
plt.ylabel("Bandwidth (-3dB) [Hz]")
plt.title("Peak Frequency vs Bandwidth (3dB) Colored by Entropy")
plt.colorbar(scatter, label="Entropy")

plt.grid(True)
plt.show()

In [ ]:
# Create the scatter plot center_frequency vs bandwidth
plt.figure()
scatter = plt.scatter(
    x=center_frequencies,
    y=bandwidths_3db,
    c=entropies,
    cmap="viridis",  
    alpha=0.7,       
)
plt.xscale('log')
plt.yscale('log')

ax = plt.gca()  # Get current axis

# Set custom tick positions and labels for x-axis
x_ticks = [100, 250, 500, 1000, 2000, 4000]
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_ticks)

# Set custom tick positions and labels for y-axis
y_ticks = [50, 100, 250, 500, 1000, 2000, 4000]
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_ticks)

# Add labels and colorbar
plt.xlabel("Center Frequency [Hz]")
plt.ylabel("Bandwidth (-3dB) [Hz]")
plt.title("Center Frequency vs Bandwidth (3dB) Colored by Entropy")
plt.colorbar(scatter, label="Entropy")

plt.grid(True)
plt.show()

In [ ]:
# Create the scatter plot spectral centroid vs spectral spread
plt.figure()
scatter = plt.scatter(
    x=centroid_frequencies,
    y=spreads,
    c=entropies,
    cmap="viridis",  
    alpha=0.7,       
)
plt.xscale('log')
plt.yscale('log')

ax = plt.gca()  # Get current axis

# Set custom tick positions and labels for x-axis
x_ticks = [100, 250, 500, 1000, 2000, 4000]
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_ticks)

# Set custom tick positions and labels for y-axis
y_ticks = [50, 100, 250, 500, 1000, 2000, 4000]
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_ticks)

# Add labels and colorbar
plt.xlabel("Spectral centroid [Hz]")
plt.ylabel("Spectral spread [Hz]")
plt.title("Spectral centroid vs spectral spread. Colored by Entropy")
plt.colorbar(scatter, label="Entropy")

plt.grid(True)
plt.show()

## Optional: run code below if you like looking at the filters! (it will result in about 700 x 2 plots...)
The plot shows:
 - The frequency (magnitude) domain plot of the filter response
 - The time domain plot of the filter response
 - In the frequency domain plot the following parts are highlighted:
    - The centroid frequency ("Center", green vertical line) 
    - 1 spectral-spread to the left of the centroid ("F_low") and 1 spectral spread to the right of the centroid ("F_high")
    - the -3 dB line. Note that for noisy signals (or at least: high entropy signals) it is not really obvious which -3 dB points to take.
 - Additionally, the list of features is plotted. It sadly lies on top of the interesting part very often :S 

In [ ]:
results, revcors_f = fu.process_data_Carney(revcors, plotting=True)